# Buổi 6 — Tư duy phản biện với dữ liệu PISA: thi thích ứng, cụm, đa kiểm định, trọng số

In [1]:
# Chạy ô này đầu tiên. Dữ liệu (PISA 2025, Việt Nam) được tải trực tiếp từ GitHub ở ô kế tiếp — không cần tải/upload tay.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf, statsmodels.api as sm
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",3)

In [2]:
# ---- TẢI DỮ LIỆU (2 bảng: cấp trường 195 dòng, cấp học sinh 7.368 dòng) ----
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
hs = pd.read_csv(RAW + "vnm_hocsinh_7368.csv")
VUNG={1:"ĐB sông Cửu Long",2:"Bắc TB, DH TB & Tây Nguyên",3:"Trung du & MN phía Bắc",4:"ĐB sông Hồng",5:"Đông Nam Bộ"}
truong["vung_ten"]=truong.vung.map(VUNG); truong["loai"]=truong.PRIVATESCH.map({1:"Công",2:"Tư"})
W=["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP","AVLRSOFT","ENCOURPG"]
print("Bảng trường:",truong.shape,"| Bảng học sinh:",hs.shape)

Bảng trường: (195, 36) | Bảng học sinh: (7368, 54)


## 1. Thi thích ứng làm điểm thô 'phẳng'

In [3]:
lin=hs[hs.SPATH==1].sci_mc.dropna(); ada=hs[hs.SPATH==2].sci_mc.dropna()
print(f"Tuyến tính: {lin.mean():.3f} (n={len(lin)}) | Thích ứng: {ada.mean():.3f} (n={len(ada)}) |",stats.ttest_ind(lin,ada,equal_var=False))

Tuyến tính: 0.456 (n=1548) | Thích ứng: 0.454 (n=4489) | TtestResult(statistic=np.float64(0.40947812920190474), pvalue=np.float64(0.682224859403098), df=np.float64(2433.009664022239))


**❓** Đường thích ứng giao câu khó cho em giỏi. Điều đó làm tỉ lệ đúng thô hội tụ về ~45%. Có thể dùng `sci_mc` để nói 'em nào giỏi hơn' không?

## 2. Thiết kế xoay vòng: không em nào làm đủ 3 lĩnh vực

In [4]:
h=hs; A=(h.sci_n>0); B=(h.math_n>0); C=(h.read_n>0)
print("Khoa học:",A.sum(),"| Toán:",B.sum(),"| Đọc:",C.sum(),"| KH+Toán:",(A&B).sum(),"| KH+Đọc:",(A&C).sum(),"| Toán+Đọc:",(B&C).sum(),"| cả 3:",(A&B&C).sum())
print(h[["sci_mc","math_mc","read_mc","ldw_mc"]].corr().round(2))

Khoa học: 6037 | Toán: 3133 | Đọc: 3161 | KH+Toán: 2258 | KH+Đọc: 2280 | Toán+Đọc: 430 | cả 3: 0
         sci_mc  math_mc  read_mc  ldw_mc
sci_mc     1.00     0.61     0.51    0.56
math_mc    0.61     1.00     0.66    0.61
read_mc    0.51     0.66     1.00    0.49
ldw_mc     0.56     0.61     0.49    1.00


## 3. Cụm: 6.037 học sinh nhưng cỡ mẫu hiệu dụng chỉ ~640 (Buổi 3, phần 4)

## 4. Đa kiểm định: 7 chỉ số × 3 lĩnh vực = 21 tương quan

In [5]:
res=[(c,o,*stats.pearsonr(truong[c],truong[o])) for c in W for o in ["sci_mean","math_mean","read_mean"]]
r=pd.DataFrame(res,columns=["chỉ số","kết quả","r","p"]); print("p<0.05:",int((r.p<.05).sum()),"/21 | p<0.05/21 (Bonferroni):",int((r.p<.05/21).sum())); r[r.p<.05].round(3)

p<0.05: 2 /21 | p<0.05/21 (Bonferroni): 0


,chỉ số,kết quả,r,p
16,AVLRSOFT,math_mean,0.149,0.038
17,AVLRSOFT,read_mean,0.143,0.047


**❓** Xác suất kỳ vọng có ≥1 p<.05 trong 21 kiểm định *nếu không có liên hệ thật* là bao nhiêu (1−0.95²¹)? Bạn có nên viết bài về 'phần mềm học tập liên quan điểm Toán' chỉ từ kết quả này?

## 5. Dự án cuối khoá
Chọn **một** câu hỏi nghiên cứu về trường học VN từ dữ liệu này, điền khung 7 mục (xem `bai_tap/RUBRIC.md`), viết 1 đoạn Kết quả thận trọng (nêu rõ: điểm thô, không có trọng số học sinh, có cụm, đa kiểm định).